# Arsenal-spillere i FPL 2025/26

Denne notebooken viser spilleroversikten, sesongstatistikk, xG/xA og kamp-for-kamp-data for Arsenal. Kjør cellene ovenfra og ned.

In [ ]:
%pip install -q pandas

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

candidates = [Path.cwd() / "data-source" / "data", Path.cwd().parent / "data-source" / "data"]
DATA_DIR = next((path for path in candidates if path.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Fant ikke data-source/data")

SEASON = "2025-26"
season_dir = DATA_DIR / SEASON
print("Datamappe:", season_dir)

## Last inn tabellene

In [ ]:
players = pd.read_csv(season_dir / "players_raw.csv")
teams = pd.read_csv(season_dir / "teams.csv")
gameweeks = pd.read_csv(season_dir / "gws" / "merged_gw.csv")
fixtures = pd.read_csv(season_dir / "fixtures.csv")

arsenal_team = teams.loc[teams["name"].eq("Arsenal")].iloc[0]
arsenal_id = arsenal_team["id"]
arsenal_players = players.loc[players["team"].eq(arsenal_id)].copy()
arsenal_gw = gameweeks.loc[gameweeks["team"].eq("Arsenal")].copy()

print(f"Arsenal har lag-ID {arsenal_id}")
print(f"{len(arsenal_players)} spillere i sesongoversikten")
print(f"{len(arsenal_gw):,} spiller-kamp-rader i gameweek-dataene")

## Arsenal-troppen
Prisfeltene i FPL-data er lagret i tideler av en million.

In [ ]:
position_names = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}
arsenal_players["name"] = (arsenal_players["first_name"] + " " + arsenal_players["second_name"]).str.strip()
arsenal_players["position"] = arsenal_players["element_type"].map(position_names)
arsenal_players["price_m"] = arsenal_players["now_cost"] / 10

squad_columns = [
    "id", "name", "web_name", "position", "price_m", "status",
    "minutes", "starts", "total_points", "points_per_game",
    "goals_scored", "assists", "clean_sheets", "bonus",
    "expected_goals", "expected_assists",
    "expected_goal_involvements", "selected_by_percent"
]
display(
    arsenal_players[squad_columns]
    .sort_values(["position", "total_points"], ascending=[True, False])
    .reset_index(drop=True)
)

## Samlet kampstatistikk per spiller
Denne tabellen summerer alle kamp-radene og inkluderer spillere som representerte Arsenal i løpet av sesongen.

In [ ]:
numeric_columns = [
    "minutes", "starts", "total_points", "goals_scored", "assists",
    "clean_sheets", "bonus", "bps", "expected_goals",
    "expected_assists", "expected_goal_involvements",
    "expected_goals_conceded"
]
for column in numeric_columns:
    arsenal_gw[column] = pd.to_numeric(arsenal_gw[column], errors="coerce").fillna(0)

summary = (
    arsenal_gw.groupby(["element", "name", "position"], as_index=False)[numeric_columns]
    .sum()
)
summary["points_per_90"] = summary["total_points"] / summary["minutes"].replace(0, pd.NA) * 90
summary["xg_per_90"] = summary["expected_goals"] / summary["minutes"].replace(0, pd.NA) * 90
summary["xa_per_90"] = summary["expected_assists"] / summary["minutes"].replace(0, pd.NA) * 90
summary["xgi_per_90"] = summary["expected_goal_involvements"] / summary["minutes"].replace(0, pd.NA) * 90

display(
    summary.sort_values("total_points", ascending=False)
    .round(2)
    .reset_index(drop=True)
)

## Ledende Arsenal-spillere

In [ ]:
for metric, title in {
    "total_points": "Flest FPL-poeng",
    "expected_goals": "Høyest xG",
    "expected_assists": "Høyest xA",
    "expected_goal_involvements": "Høyest xGI",
    "points_per_90": "Flest poeng per 90",
}.items():
    print(f"\n{title}")
    display(
        summary.loc[summary["minutes"] >= 180, ["name", "position", "minutes", metric]]
        .sort_values(metric, ascending=False)
        .head(10)
        .round(2)
        .reset_index(drop=True)
    )

## Kamp-for-kamp-data

In [ ]:
match_columns = [
    "GW", "kickoff_time", "name", "position", "opponent_team",
    "was_home", "minutes", "starts", "goals_scored", "assists",
    "expected_goals", "expected_assists", "total_points", "value"
]
display(
    arsenal_gw[match_columns]
    .sort_values(["GW", "kickoff_time", "name"])
    .reset_index(drop=True)
)

## Undersøk én spiller

In [ ]:
PLAYER_NAME = "Saka"  # Endre navnet her

player_matches = arsenal_gw.loc[
    arsenal_gw["name"].str.contains(PLAYER_NAME, case=False, na=False),
    match_columns,
].sort_values(["GW", "kickoff_time"])

print(f"Fant {len(player_matches)} kamper for søket '{PLAYER_NAME}'")
display(player_matches.reset_index(drop=True))